# Thực nghiệm chính: Phát hiện gian lận thẻ tín dụng

Notebook này điều phối toàn bộ thực nghiệm trên Kaggle. Giai đoạn hiện tại thực hiện EDA, tiền xử lý, xuất dữ liệu CSV theo từng fold và nạp lại dữ liệu để chuẩn bị cho bước huấn luyện mô hình.

Repository cần được clone vào `/kaggle/working`. Dữ liệu sẽ được tải tự động bằng API Kaggle chính thức và lưu trong `/kaggle/working/FAIR_2026_Experiment/data/Raw_data`.

## 1. Cài đặt thư viện tiền xử lý

In [ ]:
!git clone https://github.com/kohi-vip/FAIR_2026_Experiment.git

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "imblearn": "imbalanced-learn",
    "kagglehub": "kagglehub",
}
missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )
    print(f"[READY] Đã cài: {missing_packages}")
else:
    print("[READY] Các thư viện đã có sẵn; không cần cài lại.")

## 2. Xác định repository đã clone

Cell dưới đây không clone lại repository. Nó tìm `Notebook/EDA.py`, chuyển thư mục làm việc về project root và thêm project vào Python path.

In [ ]:
from pathlib import Path
import os
import sys

EXPECTED_REPOSITORY_NAME = "FAIR_2026_Experiment"
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

search_roots = [Path.cwd().resolve()]
if KAGGLE_WORKING_ROOT.is_dir():
    search_roots.insert(0, KAGGLE_WORKING_ROOT / EXPECTED_REPOSITORY_NAME)

project_candidates = []
for search_root in search_roots:
    if (search_root / "Notebook" / "EDA.py").is_file():
        project_candidates.append(search_root)

if not project_candidates and KAGGLE_WORKING_ROOT.is_dir():
    project_candidates = [
        path.parent.parent
        for path in KAGGLE_WORKING_ROOT.rglob("Notebook/EDA.py")
    ]

project_candidates = list(dict.fromkeys(path.resolve() for path in project_candidates))
if len(project_candidates) != 1:
    raise FileNotFoundError(
        "Không xác định duy nhất repository đã clone. "
        f"Các đường dẫn tìm thấy: {project_candidates}"
    )

PROJECT_ROOT = project_candidates[0]
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Kaggle runtime: {KAGGLE_WORKING_ROOT.is_dir()}")

## 3. Cấu hình lần chạy

Khi chạy trên nhiều máy độc lập, chỉ thay đổi cell cấu hình này. Mặc định chỉ tiền xử lý MLG-ULB fold 1 vì IEEE-CIS và Sparkov dùng Dense One-Hot kết hợp SMOTE, có thể cần rất nhiều RAM.

In [ ]:
# Các dataset hợp lệ: "MLG_ULB", "IEEE_CIS", "Sparkov".
# Trên mỗi máy, nên đặt EDA_DATASETS trùng với PREPROCESS_DATASETS.
EDA_DATASETS = ("MLG_ULB",)
PREPROCESS_DATASETS = ("MLG_ULB",)
FOLDS_TO_EXPORT = (1,)

SHOW_EDA_PLOTS = True
OVERWRITE_EXISTING_OUTPUTS = False
MAX_ESTIMATED_DENSE_GB = 8.0

# Fold sẽ được nạp vào RAM để bước huấn luyện model sử dụng.
TRAIN_DATASET = "MLG_ULB"
TRAIN_FOLD = 1
MLG_DUPLICATE_VARIANT = "without_duplicates"

if TRAIN_DATASET not in PREPROCESS_DATASETS:
    raise ValueError("TRAIN_DATASET phải nằm trong PREPROCESS_DATASETS.")
if TRAIN_FOLD not in FOLDS_TO_EXPORT:
    raise ValueError("TRAIN_FOLD phải nằm trong FOLDS_TO_EXPORT.")

## 4. Import pipeline, tải và kiểm tra dữ liệu Kaggle

In [ ]:
import importlib
from pathlib import Path
import sys

# Import cell vẫn tự hoạt động nếu được chạy riêng sau khi clone repository.
eda_module_candidates = []
if "PROJECT_ROOT" in globals():
    configured_module = Path(PROJECT_ROOT) / "Notebook" / "EDA.py"
    if configured_module.is_file():
        eda_module_candidates.append(configured_module)

kaggle_working_root = Path("/kaggle/working")
expected_module = (
    kaggle_working_root / "FAIR_2026_Experiment" / "Notebook" / "EDA.py"
)
if expected_module.is_file():
    eda_module_candidates.append(expected_module)
if not eda_module_candidates and kaggle_working_root.is_dir():
    eda_module_candidates.extend(kaggle_working_root.rglob("Notebook/EDA.py"))

eda_module_candidates = list(
    dict.fromkeys(path.resolve() for path in eda_module_candidates)
)
if len(eda_module_candidates) != 1:
    raise FileNotFoundError(
        "Không tìm thấy duy nhất Notebook/EDA.py trong repository đã clone. "
        f"Kết quả: {eda_module_candidates}"
    )

PROJECT_ROOT = eda_module_candidates[0].parent.parent
project_root_text = str(PROJECT_ROOT)
sys.path = [project_root_text, *[p for p in sys.path if p != project_root_text]]
importlib.invalidate_caches()
print(f"[READY] Python path đã nhận repository: {PROJECT_ROOT}")

from Notebook.EDA import (
    download_kaggle_data,
    find_data_file,
    load_processed_fold,
    run_eda,
    run_preprocessing,
)

DATASET_FILES = {
    "MLG_ULB": ("creditcard.csv",),
    "IEEE_CIS": ("train_transaction.csv", "train_identity.csv"),
    "Sparkov": ("fraudTrain.csv", "fraudTest.csv"),
}

datasets_needed = tuple(dict.fromkeys((*EDA_DATASETS, *PREPROCESS_DATASETS)))
resolved_input_files = download_kaggle_data(datasets=datasets_needed)
for dataset_name in datasets_needed:
    for filename in DATASET_FILES[dataset_name]:
        resolved_input_files[filename] = find_data_file(filename)
        print(f"[READY] {filename}: {resolved_input_files[filename]}")

## 5. Phân tích khám phá dữ liệu (EDA)

Giai đoạn này chỉ đọc và phân tích dữ liệu thô, không thay đổi dữ liệu và không huấn luyện mô hình.

In [ ]:
eda_reports = run_eda(
    datasets=EDA_DATASETS,
    show_plots=SHOW_EDA_PLOTS,
)

print(f"\n[HOÀN TẤT] Đã chạy EDA cho: {EDA_DATASETS}")

## 6. Tiền xử lý và xuất CSV

Scaler, imputer, One-Hot Encoder và SMOTE chỉ được fit trên training fold. Các CSV được ghi vào `/kaggle/working/FAIR_2026_Experiment/data/Processed_data` khi chạy trên Kaggle.

In [ ]:
artifacts = run_preprocessing(
    datasets=PREPROCESS_DATASETS,
    folds=FOLDS_TO_EXPORT,
    overwrite=OVERWRITE_EXISTING_OUTPUTS,
    max_estimated_dense_gb=MAX_ESTIMATED_DENSE_GB,
)

if not artifacts:
    raise RuntimeError("Pipeline không tạo ra artifact CSV nào.")

print(f"\n[HOÀN TẤT] Đã tạo {len(artifacts)} bộ train/validation CSV.")
for artifact in artifacts:
    variant_label = f" | variant={artifact.variant}" if artifact.variant else ""
    print(f"- {artifact.dataset} | fold={artifact.fold}{variant_label}")
    print(f"  train: {artifact.train_csv}")
    print(f"  validation: {artifact.validation_csv}")

## 7. Nạp dữ liệu đã xử lý cho bước huấn luyện

Bốn biến `X_train`, `y_train`, `X_valid`, `y_valid` là đầu vào trực tiếp cho các model ở giai đoạn tiếp theo. Validation fold không được áp dụng SMOTE.

In [ ]:
selected_variant = (
    MLG_DUPLICATE_VARIANT if TRAIN_DATASET == "MLG_ULB" else None
)
X_train, y_train, X_valid, y_valid = load_processed_fold(
    TRAIN_DATASET,
    fold=TRAIN_FOLD,
    variant=selected_variant,
)

print(f"Dataset: {TRAIN_DATASET} | fold: {TRAIN_FOLD}")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape} | y_valid: {y_valid.shape}")
print("\nPhân bố nhãn training sau SMOTE:")
print(y_train.value_counts().sort_index())
print("\nPhân bố nhãn validation gốc:")
print(y_valid.value_counts().sort_index())

## 8. Huấn luyện và đánh giá mô hình

Các cell huấn luyện model sẽ được triển khai tiếp từ `X_train`, `y_train`, `X_valid`, `y_valid`. Không dùng validation fold để fit scaler, encoder, SMOTE hoặc model.